# LeetCode Hot 100 - Day 13

## 今日主题：栈、单调栈与设计题

今天的四道题都围绕“怎么把嵌套和顺序信息存起来”：

1. 字符串解码：嵌套括号，用栈存“外面的世界”。
2. 柱状图中最大的矩形：单调栈找左右第一个更矮的柱子。
3. 数据流的中位数：两个堆分工，一类经典设计题。
4. 最小覆盖子串：滑动窗口 + 计数，加练题。

第 2 题和第 3 题都是困难题，但套路很固定，属于“背下来就能拿分”的类型。


## 今天怎么学

1. 先做字符串解码，把“遇到左括号存档、遇到右括号读档”想明白。
2. 再做柱状图最大矩形，它是 Day 10 单调栈的延续，重点理解宽度的算法。
3. 第三题是新题型：设计一个能被反复调用的类，用两个堆配合。
4. 第四题是滑动窗口的困难版，时间不够可以明天做，但思路一定要看。

最低目标：独立写出字符串解码和柱状图最大矩形。


## 今日题单

1. 字符串解码（LeetCode 394，中等，必做）
2. 柱状图中最大的矩形（LeetCode 84，困难，必做）
3. 数据流的中位数（LeetCode 295，困难，必做）
4. 最小覆盖子串（LeetCode 76，困难，加练）


## 昨日复习

先用 `day12_practice.ipynb` 重写：

1. 数组中的第 K 个最大元素：大小不超过 k 的小顶堆。
2. K 个一组翻转链表：数够 k 个再翻，`prev` 从 `group_next` 出发。

口述：为什么求前 k 大要用小顶堆？


## 题目 1 做题前先补：嵌套结构用栈“存档再读档”

看这个例子：`3[a2[c]]`。

要算出结果，你得先算里面的 `2[c]` 变成 `cc`，再把它放回外层，得到 `acc`，最后重复 3 次。

问题来了：**当你在里面处理 `2[c]` 的时候，外面还有信息要记住**——外层已经拼好的字符串 `a`，以及外层的重复次数 `3`。这两样东西等你处理完内层还要用。

正好是栈的拿手戏：

- 遇到 `[`：把“当前已经拼好的字符串”和“当前数字”压进栈（存档），然后把这两个变量清零，开始处理内层。
- 遇到 `]`：从栈里弹出一组（读档），把内层算好的字符串重复指定次数，拼到外层的字符串后面。

栈里存什么？存一个元组 `(之前的字符串, 重复次数)`。

还有一个细节：数字可能是多位的，比如 `12[a]`。所以读到数字时不能直接赋值，要用 `current_num = current_num * 10 + int(ch)` 一位一位累积。回顾一下下标专题里讲过的“从左往右累积”。


In [ ]:
# 手工数一遍 3[a2[c]] 的过程，先把每一步的“当前数字”和“当前字符串”打印出来
s = "3[a2[c]]"
current_num = 0
current_str = ""
stack = []

for ch in s:
    print("读到字符：", ch)
    if ch.isdigit():
        current_num = current_num * 10 + int(ch)
        print("    数字累积为：", current_num)
    elif ch == "[":
        stack.append((current_str, current_num))
        print("    存档：", stack[-1], "，然后清空当前变量")
        current_str = ""
        current_num = 0
    elif ch == "]":
        last_str, repeat = stack.pop()
        current_str = last_str + current_str * repeat
        print("    读档：", last_str, repeat, "，拼成：", current_str)
    else:
        current_str = current_str + ch
        print("    字符串变成：", current_str)

print("最终结果：", current_str)


# 题目 1：字符串解码

LeetCode 394. Decode String

## 题目描述（改写版）

给定一个经过编码的字符串 `s`，返回它解码后的结果。

编码规则是 `k[encoded_string]`，表示方括号里的内容重复 k 次，k 是正整数。编码可以嵌套，也可以并列。

## 输入

- `s`：长度 1 到 30，只包含数字、小写英文字母和方括号。输入保证合法。

## 输出

返回解码后的字符串。

## 示例

示例 1：`s = "3[a]2[bc]"`，结果是 `"aaabcbc"`。

示例 2：`s = "3[a2[c]]"`，结果是 `"accaccacc"`。

示例 3：`s = "2[abc]3[cd]ef"`，结果是 `"abcabccdcdcdef"`。

## 易漏细节

- 数字可能是多位，`12[a]` 要累积成 12。
- 嵌套时先算内层，靠栈的“后进先出”。
- 并列片段要按出现顺序拼接。


## 解法名称

**辅助栈（Auxiliary Stack）**，也叫“双栈/元组栈解法”。

## 暴力思路

递归：遇到 `[` 就找到它配对的 `]`，把中间那段递归求解。思路直白，但要写一个“找配对括号”的循环，边界容易出错。

## 优化思路

一遍遍历，四个变量：`current_num`、`current_str`、`stack`，外加一个字符判断。

- 数字：`current_num = current_num * 10 + int(ch)`。
- `[`：把 `(current_str, current_num)` 压栈，两个变量清零。
- 字母：直接拼到 `current_str` 后面。
- `]`：弹出 `(last_str, repeat)`，`current_str = last_str + current_str * repeat`。

时间 O(结果字符串的长度)，空间 O(嵌套层数)。

关键理解：**栈里存的是“外层还没用完的信息”**。


## 你来写：字符串解码

要求：

- 用辅助栈写，注意多位数字的累积。
- 写完用 `"3[a]2[bc]"`、`"3[a2[c]]"`、`"2[abc]3[cd]ef"`、`"12[a]"` 各跑一遍。

先在心里回答：栈里为什么要存“当前字符串”和“当前数字”两个东西？


In [ ]:
# 题目：字符串解码
# 解法：辅助栈（Auxiliary Stack）
# 输入：编码字符串 s，长度 1 到 30，只含数字、小写字母和方括号。
# 目标：把 k[子串] 展开成重复 k 次的子串，支持嵌套和并列。
# 输出：返回解码后的字符串。
# 注意：数字可能是多位，要用 current_num * 10 + int(ch) 累积；遇到 [ 存 (当前字符串, 当前数字) 并清零；遇到 ] 弹出后拼接。


def decode_string(s):
    # 在这里写你的代码
    pass


print(decode_string("3[a]2[bc]"))


In [ ]:
print(decode_string("3[a]2[bc]"))        # 期望 aaabcbc
print(decode_string("3[a2[c]]"))         # 期望 accaccacc
print(decode_string("2[abc]3[cd]ef"))    # 期望 abcabccdcdcdef
print(decode_string("12[a]"))            # 期望 aaaaaaaaaaaa
print(decode_string("abc"))              # 期望 abc


## 参考答案：字符串解码

```python
def decode_string_answer(s):
    stack = []
    current_num = 0
    current_str = ""

    for ch in s:
        if ch.isdigit():
            current_num = current_num * 10 + int(ch)
        elif ch == "[":
            stack.append((current_str, current_num))
            current_str = ""
            current_num = 0
        elif ch == "]":
            last_str, repeat = stack.pop()
            current_str = last_str + current_str * repeat
        else:
            current_str = current_str + ch

    return current_str
```

面试表达：

我用一个栈配合两个变量。遍历字符串：遇到数字就累积成完整数字，因为可能有多位；遇到左括号，就把已经拼好的字符串和当前数字压进栈，然后清零，表示“内层从零开始”；遇到右括号，弹出外层的字符串和重复次数，把内层结果重复后拼回外层；普通字母直接追加。时间 O(输出长度)，空间 O(嵌套层数)。


In [ ]:


def decode_string_answer(s):
    stack = []
    current_num = 0
    current_str = ""

    for ch in s:
        if ch.isdigit():
            current_num = current_num * 10 + int(ch)
        elif ch == "[":
            stack.append((current_str, current_num))
            current_str = ""
            current_num = 0
        elif ch == "]":
            last_str, repeat = stack.pop()
            current_str = last_str + current_str * repeat
        else:
            current_str = current_str + ch

    return current_str


print(decode_string_answer("3[a]2[bc]"))        # aaabcbc
print(decode_string_answer("3[a2[c]]"))         # accaccacc
print(decode_string_answer("2[abc]3[cd]ef"))    # abcabccdcdcdef
print(decode_string_answer("12[a]"))            # aaaaaaaaaaaa


## 题目 2 做题前先补：每根柱子负责找“自己能撑多宽”

先想清楚面积怎么算。假设我们决定：**矩形的高度就等于某根柱子的高度**，那么：

- 高度定了，宽度就是“左右两边能扩展多远”。
- 向左找第一个比它矮的柱子，向右找第一个比它矮的柱子，这两个矮柱子之间（不含它们）就是这个高度能覆盖的最大宽度。

宽度公式：`宽度 = 右下标 - 左下标 - 1`。

例如 `heights = [2, 1, 5, 6, 2, 3]`，看高度 5 这根（下标 2）：

- 左边第一个更矮的是下标 1（高度 1）；
- 右边第一个更矮的是下标 4（高度 2）；
- 宽度 = 4 - 1 - 1 = 2，面积 = 5 × 2 = 10。

那怎么快速找到左右第一个更矮的？这正是 Day 10 学过的**单调递增栈**：栈里始终保持高度递增的下标。当遇到一个更矮的柱子时，栈顶那些“比它高”的柱子就找到了自己的右边界。

再加一个小技巧：**在数组末尾放一个高度 0 的哨兵**，这样遍历到最后时，栈里的柱子会全部被弹出来结算，不用再写一段收尾代码。


In [ ]:
# 手工看一遍栈的变化
heights = [2, 1, 5, 6, 2, 3]
stack = []

for i in range(len(heights) + 1):
    if i < len(heights):
        current_height = heights[i]
    else:
        current_height = 0
        print("到达哨兵位置", i, "，开始结算栈里剩下的柱子")

    while stack and heights[stack[-1]] > current_height:
        height = heights[stack.pop()]
        if stack:
            left_index = stack[-1]
        else:
            left_index = -1
        width = i - left_index - 1
        print("    弹出高度", height, "，宽度", width, "，面积", height * width)
    if i < len(heights):
        stack.append(i)
    print("当前位置", i, "处理完，栈：", stack)


# 题目 2：柱状图中最大的矩形

LeetCode 84. Largest Rectangle in Histogram

## 题目描述（改写版）

给你一个非负整数数组 `heights`，表示一排宽度为 1 的柱子高度。请找出其中能画出的**最大矩形面积**。

矩形必须是连续的一段柱子，高度由这一段里最矮的那根决定。

## 输入

- `heights`：长度 1 到 100000，每个元素 0 到 10000。

## 输出

返回最大矩形的面积（整数）。

## 示例

示例 1：`heights = [2, 1, 5, 6, 2, 3]`，最大面积是 10。取高度 5 和 6 这两根，宽度 2，面积 10。

示例 2：`heights = [2, 4]`，最大面积是 4。取高度 2 那根，宽度 2，面积 4。

## 易漏细节

- 高度为 0 的柱子面积为 0，不影响结果，但要注意别让它把宽度算错。
- 宽度是 `右 - 左 - 1`，不是 `右 - 左`。
- 栈里存的是**下标**，因为宽度需要下标相减。
- 相等的高度不要着急弹栈（用严格大于判断），否则宽度会算小。


## 解法名称

**单调递增栈（Monotonic Increasing Stack）**。

## 暴力思路

枚举每一根柱子作为矩形高度，向左向右各扫一遍找更矮的柱子，时间 O(n²)。数据量 100000 时会超时，但面试可以先说这个思路，再优化。

## 优化思路

用单调递增栈，一次遍历解决：

1. 从左到右遍历下标，栈里存**高度递增**的下标。
2. 当前柱子比栈顶柱子矮时，栈顶柱子就找到了自己的右边界（当前下标 `i`）。把它弹出，计算面积：
   - 高度 = 它的高度；
   - 左边界 = 弹出后新的栈顶下标（没有就是 -1）；
   - 宽度 = `i - 左边界 - 1`。
3. 把所有能弹的都弹完，再把当前下标入栈。
4. 遍历到末尾时用一个高度 0 的“哨兵”触发收尾，把栈里剩下的柱子全部结算。

时间 O(n)，每个下标最多进栈出栈各一次；空间 O(n)。


## 你来写：柱状图中最大的矩形

要求：

- 用单调递增栈 + 末尾哨兵写。
- 面积公式是 `高度 × (i - 左边界 - 1)`，注意减一。
- 写完用 `[2,1,5,6,2,3]`、`[2,4]`、`[1]`、`[2,2]` 各跑一遍。

先在心里回答：为什么宽度是 `右 - 左 - 1`，那个减一是减什么？


In [ ]:
# 题目：柱状图中最大的矩形
# 解法：单调递增栈（Monotonic Increasing Stack）
# 输入：非负整数数组 heights，长度 1 到 100000，元素 0 到 10000。
# 目标：找出连续几根柱子能组成的最大矩形面积，高度由最矮的柱子决定。
# 输出：返回最大面积（整数）。
# 注意：栈里存下标且高度递增；遇到更矮的柱子就弹栈结算；宽度 = 当前下标 - 左边界下标 - 1；末尾用高度 0 的哨兵收尾。


def largest_rectangle_area(heights):
    # 在这里写你的代码
    pass


print(largest_rectangle_area([2, 1, 5, 6, 2, 3]))


In [ ]:
print(largest_rectangle_area([2, 1, 5, 6, 2, 3]))   # 期望 10
print(largest_rectangle_area([2, 4]))                # 期望 4
print(largest_rectangle_area([1]))                   # 期望 1
print(largest_rectangle_area([2, 2]))                # 期望 4
print(largest_rectangle_area([0]))                   # 期望 0
print(largest_rectangle_area([1, 1, 1, 1]))          # 期望 4


## 参考答案：柱状图中最大的矩形

```python
def largest_rectangle_area_answer(heights):
    stack = []
    max_area = 0
    n = len(heights)

    for i in range(n + 1):
        if i < n:
            current_height = heights[i]
        else:
            current_height = 0

        while stack and heights[stack[-1]] > current_height:
            height = heights[stack.pop()]
            if stack:
                left_index = stack[-1]
            else:
                left_index = -1
            width = i - left_index - 1
            area = height * width
            if area > max_area:
                max_area = area

        if i < n:
            stack.append(i)

    return max_area
```

面试表达：

我用单调递增栈，栈里存下标，对应的高度严格递增。遍历时如果当前柱子比栈顶矮，说明栈顶柱子的右边界到了，就弹出它并结算面积：高度是它的高度，左边界是弹出后新的栈顶下标，宽度是 `当前下标 - 左边界 - 1`。为了免去遍历结束后的收尾代码，我在数组末尾虚拟一个高度 0 的柱子，让栈里的元素全部被结算。每个下标最多进栈出栈一次，时间 O(n)，空间 O(n)。


In [ ]:


def largest_rectangle_area_answer(heights):
    stack = []
    max_area = 0
    n = len(heights)

    for i in range(n + 1):
        if i < n:
            current_height = heights[i]
        else:
            current_height = 0

        while stack and heights[stack[-1]] > current_height:
            height = heights[stack.pop()]
            if stack:
                left_index = stack[-1]
            else:
                left_index = -1
            width = i - left_index - 1
            area = height * width
            if area > max_area:
                max_area = area

        if i < n:
            stack.append(i)

    return max_area


print(largest_rectangle_area_answer([2, 1, 5, 6, 2, 3]))   # 10
print(largest_rectangle_area_answer([2, 4]))                # 4
print(largest_rectangle_area_answer([0]))                   # 0
print(largest_rectangle_area_answer([1, 1, 1, 1]))          # 4


## 题目 3 做题前先补：中位数和“两个堆”的分工

中位数的定义：把数据从小到大排好，正中间那个数；如果个数是偶数，取中间两个的平均值。

数据是一条一条进来的，每次都重新排序太慢。换个思路：

**把数据分成两半**：

- 较小的一半放在一个容器 `small` 里，我们希望随时能拿到它的**最大值**——所以它是**大顶堆**；
- 较大的一半放在 `large` 里，我们希望随时能拿到它的**最小值**——所以它是**小顶堆**。

只要保证两件事：

1. `small` 里的所有数都 ≤ `large` 里的所有数；
2. 两个堆的大小差不超过 1。

那么中位数就只在两个堆的**堆顶**产生：

- 两边一样多：`(small 堆顶 + large 堆顶) / 2`；
- `small` 多一个：`small` 的堆顶（总个数是奇数，中位数就是它）。

Python 没有大顶堆，用**存负数**来模拟：`small` 里存 `-num`，那么堆顶 `small[0]` 是负数中最小的，取反 `-small[0]` 就是原数里最大的。


In [ ]:
import heapq

# 演示“存负数模拟大顶堆”
small = []
for num in [3, 1, 7]:
    heapq.heappush(small, -num)
print("堆里存的是负数：", small)
print("实际最大值是：", -small[0])
heapq.heappop(small)
print("弹出一个后，剩余实际值：", [-x for x in small])


# 题目 3：数据流的中位数

LeetCode 295. Find Median from Data Stream

## 题目描述（改写版）

设计一个类 `MedianFinder`，支持两种操作：

- `addNum(num)`：从数据流中加入一个整数。
- `findMedian()`：返回目前所有数字的中位数。偶数个时返回中间两个数的平均值。

## 输入

- `addNum` 会被调用很多次，数字在 -100000 到 100000 之间。
- `findMedian` 也会被调用多次。

## 输出

- `addNum` 没有返回值。
- `findMedian` 返回浮点数。

## 示例

依次 `addNum(1)`、`addNum(2)`，`findMedian()` 返回 1.5；

再 `addNum(3)`，`findMedian()` 返回 2.0。

## 易漏细节

- 不能每次排序。
- 两个堆的大小差必须控制在 1 以内，否则中位数就不在两个堆顶了。
- 求平均要用 `/`，保证结果是浮点数。


## 解法名称

**对顶堆（Two Heaps）**。

## 暴力思路

用列表存所有数字，每次求中位数先排序。插入 O(1)，查询 O(n log n)，调用次数多时会超时。

## 优化思路

维护两个堆：`small`（大顶堆，存较小的那一半，用负数实现）和 `large`（小顶堆，存较大的那一半）。

`addNum(num)` 分三步，顺序不能乱：

1. 先把 `num` 压进 `small`；
2. 把 `small` 的堆顶（也就是较小的那一半里最大的）拿出来压进 `large`——这一步保证 `small` 里所有数都不大于 `large` 里的数；
3. 如果 `large` 比 `small` 多，就把 `large` 堆顶挪回 `small`——这一步保证 `small` 的大小 ≥ `large`，且差值不超过 1。

`findMedian()`：

- `small` 比 `large` 多：返回 `-small[0]`；
- 一样多：返回 `(-small[0] + large[0]) / 2`。

每次操作都是 O(log n)，空间 O(n)。

这种“先塞再挪”的写法很好记：**先无脑塞进小的一半，再把小的一半里最大的挪过去，最后谁多了挪回来。**


## 你来写：数据流的中位数

要求：

- 用两个堆写，类名 `MedianFinder`，方法名 `addNum` 和 `findMedian`。
- 注意 `small` 存的是负数。
- 写完用示例的 `1, 2, 1.5, 3, 2.0` 验证。

先在心里回答：为什么 `small` 要做成大顶堆？


In [ ]:
import heapq

# 题目：数据流的中位数
# 解法：对顶堆（Two Heaps）
# 输入：addNum 接收一个整数；findMedian 不接收参数。
# 目标：随时返回当前所有数字的中位数。
# 输出：addNum 无返回值；findMedian 返回浮点数（偶数个时是中间两个数的平均值）。
# 注意：small 用负数模拟大顶堆，存较小的一半；large 是小顶堆，存较大的一半；两边大小差不超过 1。


class MedianFinder:
    def __init__(self):
        # 在这里写你的代码
        pass

    def addNum(self, num):
        pass

    def findMedian(self):
        pass


finder = MedianFinder()
finder.addNum(1)
finder.addNum(2)
print(finder.findMedian())


In [ ]:
finder = MedianFinder()
finder.addNum(1)
finder.addNum(2)
print(finder.findMedian())    # 期望 1.5
finder.addNum(3)
print(finder.findMedian())    # 期望 2.0

finder2 = MedianFinder()
finder2.addNum(-1)
print(finder2.findMedian())   # 期望 -1.0
finder2.addNum(-2)
print(finder2.findMedian())   # 期望 -1.5
finder2.addNum(-3)
print(finder2.findMedian())   # 期望 -2.0

finder3 = MedianFinder()
for num in [6, 10, 2, 6, 5]:
    finder3.addNum(num)
print(finder3.findMedian())   # 期望 6.0


## 参考答案：数据流的中位数

```python
import heapq


class MedianFinderAnswer:
    def __init__(self):
        self.small = []
        self.large = []

    def addNum(self, num):
        heapq.heappush(self.small, -num)
        heapq.heappush(self.large, -heapq.heappop(self.small))
        if len(self.large) > len(self.small):
            heapq.heappush(self.small, -heapq.heappop(self.large))

    def findMedian(self):
        if len(self.small) > len(self.large):
            return -self.small[0]
        return (-self.small[0] + self.large[0]) / 2
```

面试表达：

我用两个堆把数据分成两半：`small` 是大顶堆，放较小的一半；`large` 是小顶堆，放较大的一半。插入时先把新数放进 `small`，再把 `small` 的最大值挪到 `large`，最后如果 `large` 反而比 `small` 多，就把它的最小值挪回 `small`。这样保证两个堆的大小差不超过 1，并且 `small` 里的数都不大于 `large` 里的数。求中位数时，如果 `small` 多一个就返回它的堆顶，否则返回两个堆顶的平均值。插入 O(log n)，查询 O(1)。


In [ ]:
import heapq


class MedianFinderAnswer:
    def __init__(self):
        self.small = []
        self.large = []

    def addNum(self, num):
        heapq.heappush(self.small, -num)
        heapq.heappush(self.large, -heapq.heappop(self.small))
        if len(self.large) > len(self.small):
            heapq.heappush(self.small, -heapq.heappop(self.large))

    def findMedian(self):
        if len(self.small) > len(self.large):
            return -self.small[0]
        return (-self.small[0] + self.large[0]) / 2


finder = MedianFinderAnswer()
finder.addNum(1)
finder.addNum(2)
print(finder.findMedian())    # 1.5
finder.addNum(3)
print(finder.findMedian())    # 2.0

finder3 = MedianFinderAnswer()
for num in [6, 10, 2, 6, 5]:
    finder3.addNum(num)
print(finder3.findMedian())   # 6.0


## 题目 4 做题前先补：滑动窗口怎么知道“达标了”

Day 02 学过滑动窗口求“最长子串”，那里判断条件很简单。这一题要判断的是：**窗口里是否已经包含了 t 的所有字符（含重复次数）**。

如果每次都重新数一遍窗口，太慢。技巧是用一个变量记录“**已经满足要求的字符种类数**”：

- `need`：字典，记录 t 里每个字符需要多少个。
- `window`：字典，记录当前窗口里每个字符有多少个。
- `valid`：当前有多少种字符已经**达标**（窗口里的数量 ≥ 需要的数量）。

规则：

- 右边界右移，把字符加进 `window`；如果这个字符刚好达标（数量正好等于 `need` 里的数量），`valid` 加一。
- 当 `valid` 等于 `len(need)` 时，说明窗口已经覆盖 t，这时开始收缩左边界：
  - 先试着更新最短答案；
  - 再把左边界右移，如果移走的字符恰好让它**不再达标**，`valid` 减一。

“正好达标”和“不再达标”这两处判断是关键，写错就会出现结果偏差。


In [ ]:
# 手工走一遍核心判断：只统计“达标种类数”
need = {"A": 1, "B": 1, "C": 1}
window = {}
valid = 0

for ch in "ADOBEC":
    if ch in need:
        if ch in window:
            window[ch] = window[ch] + 1
        else:
            window[ch] = 1
        if window[ch] == need[ch]:
            valid = valid + 1
    print("加入", ch, "后，window =", window, "，达标种类数 =", valid)

print("需要达标种类数 =", len(need))


# 题目 4：最小覆盖子串

LeetCode 76. Minimum Window Substring

## 题目描述（改写版）

给你两个字符串 `s` 和 `t`。请在 `s` 中找出一个**最短的连续子串**，使它包含 `t` 中的全部字符，包括重复的字符。如果找不到，返回空字符串 `""`。

## 输入

- `s`：长度 1 到 100000。
- `t`：长度 1 到 100000，都只包含大小写英文字母。

## 输出

返回最短覆盖子串；不存在时返回 `""`。

## 示例

示例 1：`s = "ADOBECODEBANC"`，`t = "ABC"`，结果是 `"BANC"`。

示例 2：`s = "a"`，`t = "a"`，结果是 `"a"`。

示例 3：`s = "a"`，`t = "aa"`，结果是 `""`。

## 易漏细节

- `t` 里重复的字符，窗口里也要有对应数量。
- 先让右边界扩张到达标，再收缩左边界找最短。
- 收缩时，只有在“这个字符本来刚好达标、移走后就不达标了”的情况下，才让 `valid` 减一。
- 返回的是空字符串，不是 `None`。


## 解法名称

**滑动窗口 + 哈希计数（Sliding Window with Hash Count）**。

## 暴力思路

枚举所有子串，逐个检查是否覆盖 `t`。子串数量是 O(n²)，每个检查 O(n)，总时间 O(n³)，完全不可行。

## 优化思路

滑动窗口：右边界负责“扩张到达标”，左边界负责“收缩到最短”。

1. 统计 `need`。
2. `right` 从左到右扫：把字符加进 `window`，如果某个字符刚好达标，`valid` 加一。
3. 当 `valid == len(need)` 时，进入内层 `while` 收缩：
   - 如果当前窗口比已知答案更短，更新答案的起点和长度；
   - 把左边界字符移出窗口；如果它移出后不达标了，`valid` 减一；
   - 左边界右移。
4. 结束后按记录的起点和长度切片返回。

时间 O(n)，左右边界各走一遍；空间 O(字符种类数)。


## 你来写：最小覆盖子串

要求：

- 用滑动窗口 + `need`、`window`、`valid` 三个东西写。
- 记录答案时存“起点 + 长度”，最后统一切片，不要每次都存字符串。
- 写完用示例的三组数据各跑一遍。

先在心里回答：`valid` 什么时候加一，什么时候减一？


In [ ]:
# 题目：最小覆盖子串
# 解法：滑动窗口 + 哈希计数（Sliding Window with Hash Count）
# 输入：字符串 s 和 t，长度都在 1 到 100000，只含大小写英文字母。
# 目标：在 s 中找最短的连续子串，使其包含 t 的所有字符（含重复次数）。
# 输出：返回最短覆盖子串；不存在时返回空字符串 ""。
# 注意：need 记录需要量，window 记录窗口内数量，valid 记录达标种类数；达标后收缩左边界；先记录起点和长度，最后切片。


def min_window(s, t):
    # 在这里写你的代码
    pass


print(min_window("ADOBECODEBANC", "ABC"))


In [ ]:
print(min_window("ADOBECODEBANC", "ABC"))   # 期望 BANC
print(min_window("a", "a"))                 # 期望 a
print(min_window("a", "aa"))                # 期望 空字符串
print(min_window("aa", "aa"))               # 期望 aa
print(min_window("ab", "b"))                # 期望 b
print(min_window("abc", "d"))               # 期望 空字符串


## 参考答案：最小覆盖子串

```python
def min_window_answer(s, t):
    if len(s) < len(t):
        return ""

    need = {}
    for ch in t:
        if ch in need:
            need[ch] = need[ch] + 1
        else:
            need[ch] = 1

    window = {}
    left = 0
    right = 0
    valid = 0
    start = 0
    length = len(s) + 1

    while right < len(s):
        ch = s[right]
        right = right + 1
        if ch in need:
            if ch in window:
                window[ch] = window[ch] + 1
            else:
                window[ch] = 1
            if window[ch] == need[ch]:
                valid = valid + 1

        while valid == len(need):
            if right - left < length:
                start = left
                length = right - left

            left_ch = s[left]
            left = left + 1
            if left_ch in need:
                if window[left_ch] == need[left_ch]:
                    valid = valid - 1
                window[left_ch] = window[left_ch] - 1

    if length == len(s) + 1:
        return ""
    return s[start:start + length]
```

面试表达：

我用滑动窗口。先用字典统计 t 里每个字符需要多少，再用另一个字典记录当前窗口的数量，并用一个变量记录已经达标的字符种类数。右边界不断扩张，把字符加入窗口，某个字符数量正好等于需要量时达标数加一。当达标数等于需要的种类数时，窗口就是可行解，这时开始收缩左边界找最短：每次先更新答案的起点和长度，再把左边界字符移出；如果移出前它是刚好达标的，移出后达标数要减一。时间 O(n)，空间 O(字符种类数)。


In [ ]:


def min_window_answer(s, t):
    if len(s) < len(t):
        return ""

    need = {}
    for ch in t:
        if ch in need:
            need[ch] = need[ch] + 1
        else:
            need[ch] = 1

    window = {}
    left = 0
    right = 0
    valid = 0
    start = 0
    length = len(s) + 1

    while right < len(s):
        ch = s[right]
        right = right + 1
        if ch in need:
            if ch in window:
                window[ch] = window[ch] + 1
            else:
                window[ch] = 1
            if window[ch] == need[ch]:
                valid = valid + 1

        while valid == len(need):
            if right - left < length:
                start = left
                length = right - left

            left_ch = s[left]
            left = left + 1
            if left_ch in need:
                if window[left_ch] == need[left_ch]:
                    valid = valid - 1
                window[left_ch] = window[left_ch] - 1

    if length == len(s) + 1:
        return ""
    return s[start:start + length]


print(min_window_answer("ADOBECODEBANC", "ABC"))   # BANC
print(min_window_answer("a", "a"))                 # a
print(min_window_answer("a", "aa"))                # 空字符串
print(min_window_answer("aa", "aa"))               # aa


# 今日小结

今天带走四个模式：

1. **嵌套结构用栈存档读档**：进括号前存“外面的状态”，出括号时读回来。
2. **单调栈求面积/宽度**：栈里存下标且高度递增，遇到更矮的柱子就结算；宽度是 `右 - 左 - 1`。
3. **对顶堆求中位数**：大顶堆存小数那一半，小顶堆存大数那一半，先塞再挪、谁多挪回来。
4. **滑动窗口**：右边界扩张到达标，左边界收缩到最短，用 `valid` 记录达标种类数。

其中第 2 题和第 3 题是困难题，但代码都很短，属于“记住套路就能写出来”的类型。


## 今日复盘区

- 字符串解码里，栈中存的两个东西分别是什么？
- 柱状图最大矩形里，为什么宽度要减 1？
- 对顶堆里，`small` 为什么要用负数？
- 最小覆盖子串里，`valid` 加一和减一的条件分别是什么？
- 今天是四道题里最难的一天，哪几道需要明天重写？

完成情况记录：

- 独立写出：
- 卡住的题：
- 明天重写：
- 完成日期：
